# ASHA Saathi: live demo on Kaggle

An **offline sense → decide → act → check agent** for rural community health workers (ASHAs), running on **Gemma 4 E2B**.

This notebook runs the exact same code as the on-device app. Gemma 4 runs locally inside this notebook through Ollama; there are no cloud LLM APIs.

> Internet is needed only to download Ollama, the model and the repo (the equivalent of installing the app). After that the agent itself makes no network calls.

**Run all cells** (a GPU T4 accelerator is recommended).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull gemma4:e2b-it-qat > /dev/null 2>&1 && ollama list

In [ ]:
!rm -rf demo-asha-saathi && git clone -q https://github.com/Yashwanthmdj/demo-asha-saathi.git
%cd demo-asha-saathi

## 1. Run the full agent on 4 field cases
For each case, watch the four phases: **SENSE** (Gemma extracts structured observations from free text, including Telugu), **DECIDE** (Gemma picks tools in a loop), **ACT** (care plan + Telugu message), **CHECK** (protocol guardrail + self-review + human handoff).

In [ ]:
!python3 run_cli.py

## 2. Offline error recovery: kill the model mid-shift
We stop the local model server. The agent must **not** crash or leave the health worker without an answer. It switches to protocol-only mode, still triages safely, and flags the case for a doctor.

In [ ]:
!pkill -f 'ollama serve'; sleep 2
!python3 run_cli.py 1

## 3. Local state: everything is in on-device SQLite
Patient history, the full audit trace of every agent step, and the referral **sync queue** that waits for connectivity.

In [ ]:
import sqlite3, pandas as pd
c = sqlite3.connect('asha.db')
display(pd.read_sql('select id, patient_id, status, triage, complaint from visits order by id desc', c))
display(pd.read_sql('select id, visit_id, priority, status, attempts from sync_queue', c))
display(pd.read_sql('select visit_id, phase, kind, substr(data,1,120) as data from trace order by id desc limit 25', c))